In [ ]:
import os
import numpy as np
import rasterio

# -----------------------------
# Config
# -----------------------------
ROOT_DIR = "/home/jovyan/data-store/forest-disturbance-stack-v3"
BIN_DIR = os.path.join(ROOT_DIR, "data", "derived", "annual_stacks", "binary")
OUT_BASE = os.path.join(BIN_DIR, "frequency")
MASK_PATH = os.path.join(ROOT_DIR, "data", "derived", "resampled", "forest_mask_30m_resampled.tif")

print("ROOT_DIR:", ROOT_DIR)
print("MASK_PATH exists?", os.path.exists(MASK_PATH), MASK_PATH)
print("BIN_DIR exists?", os.path.exists(BIN_DIR), BIN_DIR)

years = range(2000, 2021)
modes = ["any", "extreme"]

bands = {
    "wf": 1,
    "bt": 2,
    "hd": 3,
    "pd": 4
}

combos_double = {
    "wf_bt": ("wf", "bt"),
    "wf_hd": ("wf", "hd"),
    "bt_hd": ("bt", "hd"),
    "wf_pd": ("wf", "pd"),
    "bt_pd": ("bt", "pd"),
}

combos_triple = {
    "wf_bt_hd": ("wf", "bt", "hd"),
    "wf_bt_pd": ("wf", "bt", "pd"),
}

# Since max frequency is 21, uint8 is enough.
# Use 255 as nodata because valid values are 0..21.
FREQ_NODATA = 255

# -----------------------------
# Load forest mask
# -----------------------------
with rasterio.open(MASK_PATH) as msrc:
    forest = msrc.read(1) > 0
    mask_meta = msrc.meta.copy()

# -----------------------------
# Process each mode
# -----------------------------
for mode in modes:
    print(f"\n==============================")
    print(f"Processing mode: {mode.upper()}")
    print(f"==============================")

    out_dir = os.path.join(OUT_BASE, mode)
    os.makedirs(out_dir, exist_ok=True)

    counts = {
        "wf": np.zeros(forest.shape, dtype=np.uint8),
        "bt": np.zeros(forest.shape, dtype=np.uint8),
        "hd": np.zeros(forest.shape, dtype=np.uint8),
        "pd": np.zeros(forest.shape, dtype=np.uint8),
        "wf_bt": np.zeros(forest.shape, dtype=np.uint8),
        "wf_hd": np.zeros(forest.shape, dtype=np.uint8),
        "bt_hd": np.zeros(forest.shape, dtype=np.uint8),
        "wf_pd": np.zeros(forest.shape, dtype=np.uint8),
        "bt_pd": np.zeros(forest.shape, dtype=np.uint8),
        "wf_bt_hd": np.zeros(forest.shape, dtype=np.uint8),
        "wf_bt_pd": np.zeros(forest.shape, dtype=np.uint8),
    }

    meta = None

    for yr in years:
        infile = os.path.join(BIN_DIR, f"annual_stack_{mode}_{yr}.tif")
        if not os.path.exists(infile):
            print(f"  Missing {infile}, skipping")
            continue

        print(f"  Year {yr}")

        with rasterio.open(infile) as src:
            arr = src.read().astype(np.uint8)   # shape: (4, rows, cols)

            if meta is None:
                meta = src.meta.copy()
                meta.update(
                    count=1,
                    dtype=rasterio.uint8,
                    nodata=FREQ_NODATA,
                    compress="DEFLATE",
                    tiled=True,
                    BIGTIFF="YES"
                )

        # Read bands and force non-forest to 0 for counting
        wf = np.where(forest, arr[0], 0)
        bt = np.where(forest, arr[1], 0)
        hd = np.where(forest, arr[2], 0)
        pd = np.where(forest, arr[3], 0)

        # Ensure strictly binary just in case
        wf = (wf > 0).astype(np.uint8)
        bt = (bt > 0).astype(np.uint8)
        hd = (hd > 0).astype(np.uint8)
        pd = (pd > 0).astype(np.uint8)

        # Singles
        counts["wf"] += wf
        counts["bt"] += bt
        counts["hd"] += hd
        counts["pd"] += pd

        # Doubles (same-year co-occurrence)
        counts["wf_bt"] += ((wf == 1) & (bt == 1)).astype(np.uint8)
        counts["wf_hd"] += ((wf == 1) & (hd == 1)).astype(np.uint8)
        counts["bt_hd"] += ((bt == 1) & (hd == 1)).astype(np.uint8)
        counts["wf_pd"] += ((wf == 1) & (pd == 1)).astype(np.uint8)
        counts["bt_pd"] += ((bt == 1) & (pd == 1)).astype(np.uint8)

        # Triples
        counts["wf_bt_hd"] += ((wf == 1) & (bt == 1) & (hd == 1)).astype(np.uint8)
        counts["wf_bt_pd"] += ((wf == 1) & (bt == 1) & (pd == 1)).astype(np.uint8)

    # Write outputs with nodata outside forest
    for name, out_arr in counts.items():
        write_arr = out_arr.copy()
        write_arr[~forest] = FREQ_NODATA

        outfile = os.path.join(out_dir, f"{name}_{mode}_frequency.tif")
        with rasterio.open(outfile, "w", **meta) as dst:
            dst.write(write_arr, 1)

        valid = out_arr[forest]
        print(
            f"  Wrote {os.path.basename(outfile)} | "
            f"min={valid.min()} max={valid.max()} mean={valid.mean():.3f}"
        )

print("\n✅ Frequency rasters written successfully.")

ROOT_DIR: /home/jovyan/data-store/forest-disturbance-stack-v3
MASK_PATH exists? True /home/jovyan/data-store/forest-disturbance-stack-v3/data/derived/resampled/forest_mask_30m_resampled.tif
BIN_DIR exists? True /home/jovyan/data-store/forest-disturbance-stack-v3/data/derived/annual_stacks/binary

Processing mode: ANY
  Year 2000
  Year 2001
